# SkyShift on Google Colab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahadi5577/skyshift/blob/main/notebooks/skyshift_colab.ipynb)

Run the whole [SkyShift](https://github.com/Mahadi5577/skyshift) codebase in your browser: tests, the web app, the
precache, the experiments behind the design, and the online showcase build. Colab's machines sit much closer to
NASA's SPHEREx data than most home connections, so downloads are fast here.

**Use:** *Runtime > Run all*, or run the sections you need from top to bottom. A CPU runtime is enough.
Sections 1 and 2 are needed by everything else.

## 1. Get the code and install it

In [ ]:
# Clone (or update) SkyShift and install it into its own environment. A separate environment
# leaves Colab's own packages alone, so no runtime restart is needed.
import os

REPO_URL = "https://github.com/Mahadi5577/skyshift.git"
ROOT = "/content/skyshift"
if os.path.isdir(ROOT):
    !git -C {ROOT} pull -q --ff-only
else:
    !git clone -q --depth 1 {REPO_URL} {ROOT}
%cd {ROOT}

!pip -q install uv
if not os.path.exists("/content/venv/bin/python"):
    !uv venv -q --python 3.12 /content/venv
!uv pip install -q --python /content/venv/bin/python -r requirements-dev.txt -r experiments/requirements.txt
PY = "/content/venv/bin/python"
os.environ["PYTHONPATH"] = ROOT  # so your own scripts can `import backend`
# Colab sets MPLBACKEND to its notebook backend, which the scripts' environment doesn't have:
# make them draw into files instead.
os.environ["MPLBACKEND"] = "Agg"
!{PY} -c "import backend, sys; print('SkyShift', backend.__version__, '| Python', sys.version.split()[0])"

## 2. Where to keep downloads

In [ ]:
# Colab's disk is wiped when the session ends. Tick USE_DRIVE to keep the download cache and the
# Hunt database in your Google Drive instead (folder MyDrive/skyshift).
USE_DRIVE = False  # @param {type:"boolean"}

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORE = "/content/drive/MyDrive/skyshift"
else:
    STORE = "/content/skyshift-store"
os.makedirs(STORE, exist_ok=True)
os.environ["SKYSHIFT_CACHE"] = f"{STORE}/cache"
os.environ["SKYSHIFT_DATA"] = f"{STORE}/data"
print("downloads ->", os.environ["SKYSHIFT_CACHE"])

## 3. Tests
Offline tests first, then the one that asks NASA's archive (IRSA) for real.

In [ ]:
!{PY} -m pytest -q
!SKYSHIFT_NETWORK=1 {PY} -m pytest -q -k live

## 4. How fast is the data from here?
Experiment 6 times archive searches and image reads from this machine (it first lists M51's images if that hasn't been done).

In [ ]:
!cd experiments && ([ -f data/M51/index.csv ] || {PY} 01_epochs.py M51) && {PY} 06_speed.py

## 5. Run the web app
The precache downloads the guided tour and the Hunt patches first (a few minutes), so everything opens instantly.
The next cell starts the server, prints a link that opens the app in a new tab, and also shows it below.

In [ ]:
!{PY} scripts/precache.py

In [ ]:
import subprocess, time, urllib.request

def server_up():
    try:
        return urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2).read().decode()
    except Exception:
        return None

if not server_up():
    server = subprocess.Popen([PY, "-m", "uvicorn", "backend.app:app", "--host", "127.0.0.1", "--port", "8000"],
                              cwd=ROOT, stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if server_up():
            break
        time.sleep(1)
print(server_up() or "server did not start: see /content/server.log")

from google.colab import output
from google.colab.output import eval_js
print("Open the app in a new tab:", eval_js("google.colab.kernel.proxyPort(8000)"))
output.serve_kernel_port_as_iframe(8000, height=900)

To stop the server: `server.terminate()`. Its log is in `/content/server.log`.

## 6. The experiments behind the design
Each one writes its pictures and tables under `experiments/data/<target>/`. Details: `experiments/GUIDE.md`;
measured results: `experiments/FINDINGS.md`.

In [ ]:
from IPython.display import Image, display
EXP = f"{ROOT}/experiments"

def show(path):
    if os.path.exists(path):
        display(Image(path))
    else:
        print("not made (see the error above):", path)

!cd experiments && {PY} 00_check.py

### 6a. When and at which wavelengths did SPHEREx look at M51? Then blink it.

In [ ]:
!cd experiments && {PY} 01_epochs.py M51 && {PY} 02_stamps.py M51 && {PY} 03_blink.py M51
for name in ("epochs.png", "difference.png", "blink_frames.gif"):
    show(f"{EXP}/data/M51/{name}")

### 6b. Catch asteroid (10) Hygiea moving (JPL Horizons gives its position)

In [ ]:
!cd experiments && {PY} 04_asteroid.py 10
for name in ("asteroid_panels.png", "asteroid.gif"):
    show(f"{EXP}/data/ast_10/{name}")

### 6c. Plant fake movers in real frames: how faint can a mover be and still be found?

In [ ]:
!cd experiments && {PY} 01_epochs.py "180,0" && {PY} 02_stamps.py "180,0" && {PY} 07_inject.py "180,0"
show(f"{EXP}/data/180_0/inject_panels.png")
if os.path.exists(f"{EXP}/data/180_0/inject_results.csv"):
    print(open(f"{EXP}/data/180_0/inject_results.csv").read())

## 7. Build the online showcase
The same static copy that GitHub Pages serves at https://mahadi5577.github.io/skyshift/ (tour stops at every
zoom plus Hunt practice), built here and opened in a new tab.

In [ ]:
import subprocess, time
from google.colab import output

!{PY} scripts/export_static.py --out site
site = subprocess.Popen([PY, "-m", "http.server", "8001", "-d", f"{ROOT}/site"],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
from google.colab.output import eval_js
print("Open the showcase in a new tab:", eval_js("google.colab.kernel.proxyPort(8001)"))
output.serve_kernel_port_as_iframe(8001, height=900)

## 8. Your own scripts
Put scripts in a Google Drive folder (set it below) and run them with SkyShift's environment. They can
`import backend` (for example `from backend import spherex, sequences`) and use the same download cache.

In [ ]:
MY_SCRIPTS = "/content/drive/MyDrive/skyshift-scripts"  # @param {type:"string"}
SCRIPT = ""  # @param {type:"string"}

if not os.path.isdir("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
!ls -la "{MY_SCRIPTS}"
if SCRIPT:
    !cd "{MY_SCRIPTS}" && {PY} "{SCRIPT}"

## 9. Keep the experiment results
Copies `experiments/data` to `MyDrive/skyshift/experiments-data`.

In [ ]:
if not os.path.isdir("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/skyshift && cp -r {EXP}/data /content/drive/MyDrive/skyshift/experiments-data
!ls /content/drive/MyDrive/skyshift/experiments-data